# Waiting Set Stage

A `WaitingSetStage` holds agents in place when it is `ACTIVE` and lets them pass through like a waypoint when it is `INACTIVE`.
Create it with `simulation.add_waiting_set_stage(positions)`, retrieve the handle with `simulation.get_stage(id)`, and toggle `stage.state` between `jps.WaitingSetState.ACTIVE` and `jps.WaitingSetState.INACTIVE`.
This notebook starts the set active so agents gather, then deactivates it once at least 8 agents have arrived so they all proceed to the exit.

See {doc}`Object model & lifecycle </concepts/object_model>` for the full stage/journey lifecycle.

In [1]:
import pathlib

import jupedsim as jps
import shapely

# 20 m x 10 m room.
geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])

trajectory_file = pathlib.Path("waiting.sqlite")
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

# Waiting set: when ACTIVE agents hold position; when INACTIVE they pass through.
waiting_id = simulation.add_waiting_set_stage([(13, 5), (13, 5.4), (13, 4.6),
                                               (13.4, 5), (13.4, 5.4), (13.4, 4.6),
                                               (12.6, 5), (12.6, 5.4), (12.6, 4.6),
                                               (13, 5.8), (13, 4.2)])
waiting = simulation.get_stage(waiting_id)
waiting.state = jps.WaitingSetState.ACTIVE  # start ACTIVE so agents gather

exit_id = simulation.add_exit_stage([(19, 4), (20, 4), (20, 6), (19, 6)])

# Journey: waiting set → exit.
journey = jps.JourneyDescription([waiting_id, exit_id])
journey.set_transition_for_stage(waiting_id, jps.Transition.create_fixed_transition(exit_id))
journey_id = simulation.add_journey(journey)

# Place 12 agents on the left side.
n_agents = 12
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (4, 0.5), (4, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=waiting_id,
            position=position,
            radius=0.12,
        )
    )

# Once at least 8 agents have gathered, release them all by going INACTIVE.
# A flag prevents re-activation.
released = False
while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    if not released and waiting.count_waiting() >= 8:
        waiting.state = jps.WaitingSetState.INACTIVE
        released = True
    simulation.iterate()

print(f"Done in {simulation.iteration_count()} iterations. Trajectories: {trajectory_file}")

Done in 1863 iterations. Trajectories: waiting.sqlite


## Watch it

In [2]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Change the release threshold from `>= 8` to `>= 4` (or `>= 12`) and re-run. A lower threshold releases agents sooner with fewer gathered; a higher threshold makes the set hold everyone longer before they proceed to the exit.

## See also

- {doc}`Notifiable Queue Stage </notebooks/fundamentals/05_queues>` — release agents one by one from a line.
- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` — chain stages with decisions.
- {doc}`Object model & lifecycle </concepts/object_model>`.
- [jupedsim-scenarios](https://scenarios.jupedsim.org) — parameter sweeps and Monte-Carlo runs.